In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import duckdb
import os

class DataLoader():

    def __init__(self, base_url='gs://clusterdata_2019_a', threads=None, max_workers=None):
        self.base_url = base_url
        # max_workers × threads ≈ 2x number of CPU cores; assuming 10 cores
        num_cpus = os.cpu_count()  # logical cores
        self.threads = threads or max(1, num_cpus // 4)
        self.max_workers = max_workers or max(1, num_cpus // self.threads)


    def parallel_load(self, fn, shards=0, max_workers=None, batch_size=10, **kwargs) -> pd.DataFrame:
        '''
        function to run another function (that returns a dataframe) across concurrent futures.
            args :
                fn : function which will be executed by concurrent futures
                shards : number of shards we want to read from the gcs bucket
                max-workers : number of parallel workers
                batch_size : read shards in batches of batch_size\
        '''
        max_workers = max_workers or self.max_workers
        batches = list(range(0, shards, batch_size))  # [0, 10, 20, ...]
        results = [None] * len(batches)
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futs = {
                ex.submit(fn, start, min(start + batch_size - 1, shards - 1), **kwargs): i
                for i, start in enumerate(batches)
            }
            for fut in as_completed(futs):
                i = futs[fut]
                results[i] = fut.result()
                print(f"batch {i+1}/{len(batches)}")
        return pd.concat(results, ignore_index=True)


    def dck_db_load(
                      self,
                      shard_range_start=None,
                      shard_range_end=None,
                      query=None,
                      cols=None,
                      target_collection_ids=None,
                      target_instance_indexes=None,
                      target_pairs=None
                      ) -> pd.DataFrame:

        '''
        function to initialize a duckdb connection and query shards between 'shard_range_start' and 'shard_range_end' from gcs bucket.
        args:
            shard_range_start : start reading shards from here
            shard_range_end : stop reading here
            query : string containing the query to run
            cols : string containing comma separated cols to pull (goes inside the query)
            target_collection_ids : string containing comma separated collection ids to filter on (goes inside the query)
            target_instance_indexes : string containing comma separated instance idxs to filter on (goes inside the query) // this might probably be removed
            target_pairs : string containing comma separated pairs of (collection_id, instance_index) to filter on (goes inside the query)
        '''
        assert shard_range_start is not None, "shard_range_start cannot be None"
        assert shard_range_end is not None, "shard_range_end cannot be None"
        assert query is not None, "query cannot be None"
        assert cols is not None, "cols cannot be None"
        if (query == 'instance_usage') | (query == 'collection_events'):
          assert target_collection_ids is not None, "collection_ids cannot be None"
        if query == 'instance_usage':
          assert target_instance_indexes is not None, "instance_indexes cannot be None"
        if query == 'instance_usage_1':
          assert target_pairs is not None, "instance_pairs cannot be None"

        con = duckdb.connect()
        con.sql("INSTALL httpfs; LOAD httpfs;")
        con.sql(f"SET threads = {self.threads};")

        try:
            dfs=[]
            print(f"processing {shard_range_start} to {shard_range_end}...")
            for i in range(shard_range_start,shard_range_end+1):
                char = str(i).zfill(5)
                print(f"processing {char}...")

                query_set = {
                    'collection_events' : f"""
                            SELECT {cols}
                            FROM read_parquet('{self.base_url}/collection_events-0000000{char}.parquet.gz')
                            WHERE collection_id IN ({target_collection_ids})
                            LIMIT 10000
                        """,
                    'instance_events' : f"""
                            SELECT {cols}
                            FROM read_parquet('{self.base_url}/instance_events-0000000{char}.parquet.gz')
                            WHERE collection_id IN ({target_collection_ids})
                            --LIMIT 10000
                        """,
                    'instance_usage' : f"""
                            SELECT {cols}

                            FROM read_parquet('{self.base_url}/instance_usage-0000000{char}.parquet.gz')
                            WHERE collection_id IN ({target_collection_ids})
                            AND instance_index IN ({target_instance_indexes})
                            ORDER BY collection_id, instance_index, start_time
                            --LIMIT 10000
                        """,
                    'instance_usage_1' : f"""
                            SELECT {cols}
                            FROM read_parquet('{self.base_url}/instance_usage-0000000{char}.parquet.gz')
                            WHERE (collection_id, instance_index) IN ({target_pairs})
                            ORDER BY collection_id, instance_index, start_time
                            --LIMIT 10000"""
                }

                #change query here
                df = con.sql(query_set[query]).df()
                dfs.append(df)
        finally:
            con.close()

        return pd.concat(dfs, ignore_index=True)

In [5]:
DATA_DIR = "/content/drive/MyDrive/Bayesian Final Project/DATA"
coll_events = pd.read_parquet(f"{DATA_DIR}/collections_per_priority_parquet.parquet")
coll_events.head(1).T

,0
collection_id,382836131204
user,unknown_user
priority,107
scheduling_class,0
num_instances,9
evictions,0
failures,1
kills,0
finished,False
outcome,failed


In [10]:
_collection_ids_unique = coll_events.collection_id.unique()
print(len(_collection_ids_unique))
target_collection_ids = ",".join(_collection_ids_unique.astype(str))
target_collection_ids

1200


'382836131204,383851959840,381105478433,380756707566,399482865043,399595493272,399280623256,381985747703,383224655865,380107214785,383175726451,399880512375,382871403557,383288833095,383502500032,382339850817,383245376450,399280603720,383935187161,396209193336,384616375362,383853127716,399880499837,381151741072,383102456494,19542402683,383584132824,380128692299,383168330598,399880611953,399280626615,383162438237,380756796946,382834274636,383119912850,383049387956,385644689424,380190242235,381589971505,383057854399,382178778804,380756847257,395095968041,393986415502,381781266242,383097010648,399281639261,383007226500,383220010638,384347908201,383934960446,383580277494,382182022963,382985046393,382975382187,383109339182,383102454915,383934961871,383596564347,399595450937,383167913696,383187485657,381781273742,382186799401,383190825959,383057481410,394286938508,399880782949,383149589472,399280604220,399880682150,399880602964,382916261356,382866558237,384345846007,381897833000,399280603892

In [11]:
# sample : pulls 10 shards, in default batches of 10, gets cols collection_id and priority from collection id
dl = DataLoader()
df = dl.parallel_load(dl.dck_db_load, shards=10, query='instance_events', cols='collection_id, instance_index, time, type', target_collection_ids=target_collection_ids)

processing 0 to 9...
processing 00000...
processing 00001...
processing 00002...
processing 00003...
processing 00004...
processing 00005...
processing 00006...
processing 00007...
processing 00008...
processing 00009...
batch 1/1


In [12]:
df.to_parquet(f"{DATA_DIR}/instance_events_training01_parquet.parquet")

In [13]:
df = pd.read_parquet(f"{DATA_DIR}/instance_events_training01_parquet.parquet")

In [14]:
df.shape

(125586, 4)

In [15]:
df.groupby(['collection_id', 'instance_index','time']).size()

collection_id  instance_index  time         
9941075899     4               55847243279      1
               5               1230403923656    1
               8               586839859943     1
                               2616883466849    1
               9               405828165600     1
                                               ..
400460420344   446             2677711878678    1
               478             2677700156461    1
               480             2677736479379    1
               485             2677712208676    1
400463175153   4               2678224046330    1
Length: 124850, dtype: int64

In [27]:
df2 = df[["collection_id", "instance_index", "time"]].copy()

# Ensure integer dtype (helps speed + avoids object groupby)
df2["collection_id"] = pd.to_numeric(df2["collection_id"], errors="coerce").astype("Int64")
df2["instance_index"] = pd.to_numeric(df2["instance_index"], errors="coerce").astype("Int64")
df2["time"] = pd.to_numeric(df2["time"], errors="coerce").astype("Int64")
df2 = df2.dropna()

# -----------------------------
# 1) detect unix unit (override if you already know it)
# -----------------------------
mx = int(df2["time"].max())

time_unit = "us"


print("Detected time unit:", time_unit, "| max time:", mx)

# -----------------------------
# 2) aggregate per (collection_id, instance_index)
# -----------------------------
agg = (
    df2.groupby(["collection_id", "instance_index"], sort=False, observed=True)["time"]
       .agg(time_min="min", time_max="max", n_rows="size")
       .reset_index()
)

# span in raw units + as timedelta
agg["span_units"] = agg["time_max"] - agg["time_min"]
agg["span"] = pd.to_timedelta(agg["span_units"].astype("int64"), unit=time_unit)

# -----------------------------
# 3) top 10 by longest duration
# -----------------------------
top50 = (
    agg.sort_values(["span_units", "n_rows"], ascending=[False, False])
       #.head(1000)
       .reset_index(drop=True)
)

# Optional: nicer HH:MM:SS (includes days if needed)
top50["span_hhmmss"] = top50["span"].astype(str)

# Select the columns you asked for
top50 = top50[["collection_id", "instance_index", "n_rows", "span", "span_units" ]] #, "span_hhmmss", "time_min", "time_max"]]

display(top50.head(1000))

Detected time unit: us | max time: 2678983818889


,collection_id,instance_index,n_rows,span,span_units
0,330587125888,29,37,31 days 00:07:42.059528,2678862059528
1,280487004463,0,538,31 days 00:04:51.905633,2678691905633
2,259837769991,0,547,30 days 23:58:53.361288,2678333361288
3,330587125888,1360,19,30 days 23:56:56.216278,2678216216278
4,301297312669,0,536,30 days 23:55:01.576675,2678101576675
...,...,...,...,...,...
995,330587125888,1444,11,29 days 03:22:28.708453,2517748708453
996,330587125888,857,15,29 days 03:16:10.959154,2517370959154
997,330587125888,2373,7,29 days 03:15:31.543702,2517331543702
998,330587125888,1350,10,29 days 03:15:02.723725,2517302723725


In [28]:
top50['span_units'].describe()

,span_units
count,41063.0
mean,187344746780.158569
std,607692517941.44519
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,2678862059528.0


In [39]:
_top50 = top50[top50['span_units'] > 0].copy()
_top50.shape

(7061, 5)

In [40]:
_jdf = pd.merge(_top50, coll_events, on=['collection_id'], how='left')

In [41]:
#_jdf.head(1).T
_jdf.groupby(['priority_bin', 'outcome']).size()

priority_bin  outcome    
2             failed          919
              finished       2580
              manual_kill     366
              other            33
3             failed          333
              finished         48
              manual_kill     476
              other            27
4             failed          402
              finished        114
              manual_kill    1737
              other            26
dtype: int64

In [42]:
sampled_jdf = _jdf.groupby(['outcome', 'priority_bin']).apply(lambda x: x.sample(n=min(100, len(x)), random_state=42)).reset_index(drop=True)
display(sampled_jdf.head())

/tmp/ipykernel_10927/1783610777.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_jdf = _jdf.groupby(['outcome', 'priority_bin']).apply(lambda x: x.sample(n=min(100, len(x)), random_state=42)).reset_index(drop=True)


,collection_id,instance_index,n_rows,span,span_units,user,priority,scheduling_class,num_instances,evictions,failures,kills,finished,outcome,priority_bin
0,399280623256,758,2,0 days 03:56:15.838407,14175838407,unknown_user,103,1,88,0,19,0,False,failed,2
1,399280623256,1485,2,0 days 03:26:12.150615,12372150615,unknown_user,103,1,88,0,19,0,False,failed,2
2,399280623256,108,3,0 days 02:14:07.383280,8047383280,unknown_user,103,1,88,0,19,0,False,failed,2
3,399281639261,311,2,0 days 02:06:56.536742,7616536742,unknown_user,103,1,98,0,21,0,False,failed,2
4,399280605712,979,2,0 days 00:20:30.691075,1230691075,unknown_user,103,1,100,0,24,0,False,failed,2


In [43]:
sampled_jdf.groupby(['priority_bin', 'outcome']).size()

priority_bin  outcome    
2             failed         100
              finished       100
              manual_kill    100
              other           33
3             failed         100
              finished        48
              manual_kill    100
              other           27
4             failed         100
              finished       100
              manual_kill    100
              other           26
dtype: int64

In [44]:
_target_collection_ids = ",".join(sampled_jdf.collection_id.astype(str).to_list())
_target_instance_indexes = ",".join(sampled_jdf.instance_index.astype(str).to_list())

In [45]:
pairs = list(
    zip(
        sampled_jdf["collection_id"],
        sampled_jdf["instance_index"]
    )
)
pair = list(set(pairs))
target_pairs = ", ".join(
    f"({int(cid)}, {int(iid)})"
    for cid, iid in pairs
)

In [46]:
cols = '''start_time,
        collection_id,
        instance_index,
        average_usage.cpus as average_usage_cpus,
        average_usage.memory as average_usage_memory,
        maximum_usage.cpus as maximum_usage_cpus,
        maximum_usage.memory as maximum_usage_memory,
        cycles_per_instruction,
        random_sample_usage.cpus as random_sample_usage_cpus,
        random_sample_usage.memory as random_sample_usage_memory,
        assigned_memory,
        page_cache_memory,
        memory_accesses_per_instruction,
        cpu_usage_distribution,
        tail_cpu_usage_distribution'''


In [47]:
dl = DataLoader()

#_instance_usages_df = dl.parallel_load(
    #dl.dck_db_load, shards=1000,
    #batch_size=100, query='instance_usage',
    #cols= cols,
    #target_collection_ids=_target_collection_ids,
    #target_instance_indexes=_target_instance_indexes)

_instance_usages_df = dl.parallel_load(
    dl.dck_db_load, shards=500,
    batch_size=50, query='instance_usage_1',
    cols= cols,
    target_collection_ids=None,
    target_instance_indexes=None,
    target_pairs = target_pairs)

processing 0 to 49...
processing 00000...
processing 50 to 99...
processing 00050...
processing 100 to 149...
processing 00100...
processing 150 to 199...
processing 00150...
processing 00051...
processing 00001...
processing 00151...
processing 00101...
processing 00052...
processing 00002...
processing 00152...
processing 00102...
processing 00003...
processing 00053...
processing 00153...
processing 00103...
processing 00004...
processing 00154...
processing 00054...
processing 00104...
processing 00005...
processing 00155...
processing 00105...
processing 00055...
processing 00006...
processing 00156...
processing 00056...
processing 00106...
processing 00007...
processing 00157...
processing 00057...
processing 00107...
processing 00008...
processing 00158...
processing 00058...
processing 00108...
processing 00009...
processing 00059...
processing 00159...
processing 00109...
processing 00010...
processing 00110...
processing 00160...
processing 00060...
processing 00011...
proce

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

processing 00398...
processing 00347...
processing 00248...
processing 00297...
processing 00348...
processing 00399...
processing 00249...
processing 00298...
processing 00349...
batch 8/10
processing 400 to 449...
processing 00400...
batch 5/10
processing 450 to 499...
processing 00450...
processing 00299...
processing 00401...
batch 7/10
batch 6/10
processing 00451...
processing 00402...
processing 00452...
processing 00403...
processing 00453...
processing 00404...
processing 00454...
processing 00405...
processing 00455...
processing 00406...
processing 00456...
processing 00407...
processing 00457...
processing 00408...
processing 00458...
processing 00409...
processing 00459...
processing 00410...
processing 00460...
processing 00411...
processing 00461...
processing 00412...
processing 00462...
processing 00413...
processing 00463...
processing 00414...
processing 00464...
processing 00415...
processing 00465...
processing 00466...
processing 00416...
processing 00467...
proces

In [48]:
_instance_usages_df.groupby(['collection_id', 'instance_index']).size().describe()

,0
count,712.000000
mean,219.765449
std,602.676410
min,1.000000
25%,6.000000
50%,46.000000
75%,188.250000
max,4793.000000


In [36]:
_instance_usages_df.to_parquet(f"{DATA_DIR}/instance_usage_training01_parquet.parquet")

In [49]:
_tmp_iu = _instance_usages_df.groupby(['collection_id', 'instance_index']).size().reset_index()

In [53]:
def average_gap_per_collection_instance(df):
    df = df.sort_values(['collection_id', 'instance_index', 'start_time']).copy()
    df['next_start_time'] = (
        df.groupby(['collection_id', 'instance_index'])['start_time']
          .shift(-1)
    )
    df['gap_seconds'] = (
        (df['next_start_time'] - df['start_time']) / 1e6
    )
    result = (
        df.groupby(['collection_id', 'instance_index'])['gap_seconds']
          .mean()
          .reset_index(name='avg_gap_seconds')
    )
    return result

In [51]:
_res = average_gap_per_collection_instance(_instance_usages_df)

In [52]:
_res.avg_gap_seconds.describe()

,avg_gap_seconds
count,647.000000
mean,13115.938993
std,33069.781848
min,1.000000
25%,2372.533333
50%,5632.000000
75%,9696.511628
max,439888.000000


In [31]:
_res.head(1).T

,0
collection_id,9.941076e+09
instance_index,8.000000e+00
avg_gap_seconds,3.324800e+04


In [32]:
_jdf1 = pd.merge(_res, sampled_jdf, on=['collection_id', 'instance_index'], how='left')

In [33]:
_jdf1.groupby(['priority_bin', 'outcome']).size()

priority_bin  outcome    
2             failed          8
              finished        8
              manual_kill     8
3             failed         10
              finished        1
              manual_kill     8
              other           3
4             failed          8
              manual_kill    10
              other           3
dtype: int64